In [1]:
from pyspark.sql import SparkSession

spark = SparkSession. \
builder. \
appName("week7-lesson-7"). \
config("spark.sql.warehouse.dir", f"/user/itv024128/warehouse"). \
enableHiveSupport(). \
master('yarn'). \
getOrCreate()
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [2]:
# /public/trendytech/orders/orders_1gb.csv

In [3]:
orderSchema = StructType([
    StructField("order_id",LongType()),
    StructField("order_date",DateType()),
    StructField("cust_id",LongType()),
    StructField("status",StringType()),
])

In [4]:
df1 = spark.read.schema(orderSchema).option("header","false").csv('/public/trendytech/orders/orders_1gb.csv')

### By default if we dont specify a format, spark tables will be stored as highly optimised PARQUET files

In [5]:
# [itv024128@g02 ~]$ hadoop fs -ls -h warehouse/itv024128.db/orders_2
# Found 10 items
# -rw-r--r--   3 itv024128 supergroup          0 2026-07-21 13:47 warehouse/itv024128.db/orders_2/_SUCCESS
# -rw-r--r--   3 itv024128 supergroup     13.3 M 2026-07-21 13:47 warehouse/itv024128.db/orders_2/part-00000-c7c71943-5df0-4791-80cb-d5e7923f8efd-c000.snappy.parquet
# -rw-r--r--   3 itv024128 supergroup     13.3 M 2026-07-21 13:47 warehouse/itv024128.db/orders_2/part-00001-c7c71943-5df0-4791-80cb-d5e7923f8efd-c000.snappy.parquet
# -rw-r--r--   3 itv024128 supergroup     13.3 M 2026-07-21 13:47 warehouse/itv024128.db/orders_2/part-00002-c7c71943-5df0-4791-80cb-d5e7923f8efd-c000.snappy.parquet
# -rw-r--r--   3 itv024128 supergroup     13.3 M 2026-07-21 13:47 warehouse/itv024128.db/orders_2/part-00003-c7c71943-5df0-4791-80cb-d5e7923f8efd-c000.snappy.parquet
# -rw-r--r--   3 itv024128 supergroup     13.3 M 2026-07-21 13:47 warehouse/itv024128.db/orders_2/part-00004-c7c71943-5df0-4791-80cb-d5e7923f8efd-c000.snappy.parquet
# -rw-r--r--   3 itv024128 supergroup     13.3 M 2026-07-21 13:47 warehouse/itv024128.db/orders_2/part-00005-c7c71943-5df0-4791-80cb-d5e7923f8efd-c000.snappy.parquet
# -rw-r--r--   3 itv024128 supergroup     13.3 M 2026-07-21 13:47 warehouse/itv024128.db/orders_2/part-00006-c7c71943-5df0-4791-80cb-d5e7923f8efd-c000.snappy.parquet
# -rw-r--r--   3 itv024128 supergroup     13.3 M 2026-07-21 13:47 warehouse/itv024128.db/orders_2/part-00007-c7c71943-5df0-4791-80cb-d5e7923f8efd-c000.snappy.parquet
# -rw-r--r--   3 itv024128 supergroup      5.3 M 2026-07-21 13:47 warehouse/itv024128.db/orders_2/part-00008-c7c71943-5df0-4791-80cb-d5e7923f8efd-c000.snappy.parquet

In [6]:
spark.sql("drop table itv024128.orders_2")

""


In [7]:
df1.write.saveAsTable("itv024128.orders_2") ## .format('csv') not added

In [8]:
spark.sql("select count(*) from itv024128.orders_2")

count(1)
25831125


### Even without cache, the count query ran quick as Parquet stores count in meta data. No need to read entire files

In [9]:
spark.sql("use itv024128")
spark.sql("show tables")

database,tableName,isTemporary
itv024128,groceries,false
itv024128,groceries_ext,false
itv024128,groceries_ext_json,false
itv024128,groceries_json,false
itv024128,orders1gb,false
itv024128,orders1gb_ext,false
itv024128,orders_2,false
itv024128,orders_ext,false


In [10]:
spark.sql("cache table itv024128.orders_2")

""


### But if we cache the parq table , and the take a count, spark has to go thru entire data in memory/cache to get the count. This may be slower that reading from parq files

In [11]:
spark.sql("select count(*) from itv024128.orders_2")

count(1)
25831125


## PERSIST

In [12]:
from pyspark.storagelevel import StorageLevel

In [ ]:
df1.persist(StorageLevel(True,False,False,False,1))   ## takes 5 args. 1-Disk, 2-Memory  3-Off Heap   4-Deserialized/Serialised  5-Number of replicas

In [16]:
df1.count()

25831125

In [17]:
df1.count()

25831125

In [ ]:
df1.unpersist()

In [ ]:
df1.persist(StorageLevel(True,True,False,True,1))

In [23]:
df1.count()

25831125

In [ ]:
df1.persist(StorageLevel(False,False,True,False,1))

In [30]:
## another syntax

In [ ]:
df1.persist(StorageLevel.OFF_HEAP)